# MadenGuard AI - Colab LiDAR Pipeline

Bu notebook buyuk LAS / parca LAS dosyalarini Colab veya server ortaminda chunk tabanli islemek icin hazirlanmistir.

Amac:

```text
LAS / parca LAS dosyalari
-> nokta bulutu okuma
-> mumkun oldugunca yuksek detayli downsample
-> hafif PLY uretimi
-> X-Y segment cikarimi
-> geometri riski
-> mine graph JSON
```

Onemli: Bu notebook `laspy.read(...)` kullanmaz. LAS dosyalari parca parca okunur.

## Cikti Stratejisi

Daha net maden goruntusu icin birden fazla PLY uretilecek:

```text
tunnel_preview_500k.ply          -> hizli frontend/demo modeli
tunnel_downsampled.ply           -> 1M noktalik daha net frontend modeli
```

Frontend icin varsayilan hedef 500k ve 1M nokta seviyesidir. Daha detayli CloudCompare gorseli gerekiyorsa notebook icindeki kalite ayarlarina ek bir 5M veya 10M cikti eklenebilir.

In [ ]:
!pip install -q laspy numpy pandas networkx tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Yol Ayarlari

`BASE_DIR` degerini Google Drive icindeki proje konumuna gore duzenle.

Ornek:

```text
/content/drive/MyDrive/MadenGuardAI
```

In [ ]:
from pathlib import Path

BASE_DIR = Path('/content/drive/MyDrive/MadenGuardAI')

# Bu notebook varsayilan olarak tek MediumRes LAS dosyasini kullanir.
# Parca LAS dosyalari kullanilacaksa SINGLE_LAS_PATH = None yapip INPUT_DIR yolunu doldur.
INPUT_DIR = None

# Colab Drive icinde beklenen dosya konumu:
# MyDrive/MadenGuardAI/data_raw/subt_las/Tunnel_Circuit_MediumRes_Scan_EX_Frame.las
SINGLE_LAS_PATH = BASE_DIR / 'data_raw' / 'subt_las' / 'Tunnel_Circuit_MediumRes_Scan_EX_Frame.las'

OUTPUT_DIR = BASE_DIR / 'backend' / 'data_processed'
POINTCLOUD_DIR = OUTPUT_DIR / 'pointcloud'
SEGMENT_DIR = OUTPUT_DIR / 'segments'
GRAPH_DIR = OUTPUT_DIR / 'graph'
RISK_DIR = OUTPUT_DIR / 'risk'
SUMMARY_DIR = BASE_DIR / 'outputs'

POINTCLOUD_DIR.mkdir(parents=True, exist_ok=True)
SEGMENT_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
RISK_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_DIR:', BASE_DIR)
print('SINGLE_LAS_PATH:', SINGLE_LAS_PATH)
print('LAS exists:', SINGLE_LAS_PATH.exists())
print('INPUT_DIR:', INPUT_DIR)

## Kalite Ayarlari

`QUALITY_LEVELS` icindeki hedefler frontend'de kullanilacak PLY dosyalarini belirler. Bu gorev icin 500k ve 1M nokta seviyeleri yeterlidir.

Not: PLY dosyalari binary yazildigi icin ASCII PLY'ye gore daha kucuk ve hizlidir.

In [ ]:
CHUNK_SIZE = 500_000
GRID_SIZE_METERS = 25.0
NARROW_PASSAGE_WIDTH_M = 8.0

# Daha net CloudCompare goruntusu icin high_detail hedefini artirabilirsin.
QUALITY_LEVELS = {
    'preview_500k': 500_000,
    'frontend_1m': 1_000_000,
}

PLY_OUTPUT_FILENAMES = {
    'preview_500k': 'tunnel_preview_500k.ply',
    'frontend_1m': 'tunnel_downsampled.ply',
}

# Tarayici/Three.js icin 500k hizli demo, 1M ise daha net demo olarak kullanilabilir.
# Daha detayli CloudCompare gorseli gerekirse buraya 5M veya 10M seviyesinde ek bir cikti eklenebilir.
print(QUALITY_LEVELS)

In [ ]:
import json
import math
import struct
from dataclasses import asdict, dataclass
from typing import Iterable

import laspy
import networkx as nx
import numpy as np
from tqdm.auto import tqdm


@dataclass
class Bounds:
    min_x: float = math.inf
    max_x: float = -math.inf
    min_y: float = math.inf
    max_y: float = -math.inf
    min_z: float = math.inf
    max_z: float = -math.inf
    point_count: int = 0

    def update_from_header(self, header) -> None:
        self.min_x = min(self.min_x, float(header.mins[0]))
        self.min_y = min(self.min_y, float(header.mins[1]))
        self.min_z = min(self.min_z, float(header.mins[2]))
        self.max_x = max(self.max_x, float(header.maxs[0]))
        self.max_y = max(self.max_y, float(header.maxs[1]))
        self.max_z = max(self.max_z, float(header.maxs[2]))
        self.point_count += int(header.point_count)


def find_las_paths() -> list[Path]:
    if SINGLE_LAS_PATH is not None:
        path = Path(SINGLE_LAS_PATH)
        if not path.exists():
            raise FileNotFoundError(path)
        return [path]

    paths = sorted(INPUT_DIR.glob('*.las'))
    if not paths:
        raise FileNotFoundError(f'No LAS files found in {INPUT_DIR}')
    return paths


def inspect_headers(paths: list[Path]) -> Bounds:
    bounds = Bounds()
    for path in paths:
        with laspy.open(path) as reader:
            bounds.update_from_header(reader.header)
    return bounds


def compute_stride(total_points: int, target_points: int) -> int:
    return max(1, math.ceil(total_points / target_points))


def count_sampled_points(paths: list[Path], stride: int) -> int:
    total = 0
    global_index = 0
    for path in paths:
        with laspy.open(path) as reader:
            n = int(reader.header.point_count)
        first = (-global_index) % stride
        if first < n:
            total += 1 + ((n - 1 - first) // stride)
        global_index += n
    return total


def write_binary_ply_header(handle, vertex_count: int) -> None:
    header = (
        'ply\n'
        'format binary_little_endian 1.0\n'
        f'element vertex {vertex_count}\n'
        'property float x\n'
        'property float y\n'
        'property float z\n'
        'end_header\n'
    )
    handle.write(header.encode('ascii'))


def save_json(path: Path, data) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding='utf-8')

In [ ]:
las_paths = find_las_paths()
bounds = inspect_headers(las_paths)

origin = {
    'x': (bounds.min_x + bounds.max_x) / 2,
    'y': (bounds.min_y + bounds.max_y) / 2,
    'z': (bounds.min_z + bounds.max_z) / 2,
}

print('LAS file count:', len(las_paths))
print('Total points:', f'{bounds.point_count:,}')
print('Bounds:', asdict(bounds))
print('Origin/global shift:', origin)

save_json(POINTCLOUD_DIR / 'global_shift.json', {
    'origin': origin,
    'bounds': asdict(bounds),
    'note': 'Viewer coordinates are stored after subtracting origin.'
})

## Segment Yardimci Fonksiyonlari

Segmentleme X-Y grid uzerinden yapilir. Z ekseni tavan/taban/yukseklik farki ve geometri riski icin kullanilir.

In [ ]:
def update_segments(segments: dict, x: np.ndarray, y: np.ndarray, z: np.ndarray) -> None:
    gx = np.floor(x / GRID_SIZE_METERS).astype(np.int64)
    gy = np.floor(y / GRID_SIZE_METERS).astype(np.int64)
    keys = np.column_stack([gx, gy])
    unique_keys, inverse = np.unique(keys, axis=0, return_inverse=True)

    counts = np.bincount(inverse)
    min_x = np.full(len(unique_keys), np.inf)
    max_x = np.full(len(unique_keys), -np.inf)
    min_y = np.full(len(unique_keys), np.inf)
    max_y = np.full(len(unique_keys), -np.inf)
    min_z = np.full(len(unique_keys), np.inf)
    max_z = np.full(len(unique_keys), -np.inf)

    np.minimum.at(min_x, inverse, x)
    np.maximum.at(max_x, inverse, x)
    np.minimum.at(min_y, inverse, y)
    np.maximum.at(max_y, inverse, y)
    np.minimum.at(min_z, inverse, z)
    np.maximum.at(max_z, inverse, z)

    for i, key in enumerate(unique_keys):
        k = (int(key[0]), int(key[1]))
        seg = segments.setdefault(k, {
            'point_count': 0,
            'min_x': math.inf,
            'max_x': -math.inf,
            'min_y': math.inf,
            'max_y': -math.inf,
            'min_z': math.inf,
            'max_z': -math.inf,
        })
        seg['point_count'] += int(counts[i])
        seg['min_x'] = min(seg['min_x'], float(min_x[i]))
        seg['max_x'] = max(seg['max_x'], float(max_x[i]))
        seg['min_y'] = min(seg['min_y'], float(min_y[i]))
        seg['max_y'] = max(seg['max_y'], float(max_y[i]))
        seg['min_z'] = min(seg['min_z'], float(min_z[i]))
        seg['max_z'] = max(seg['max_z'], float(max_z[i]))


def finalize_segments(raw_segments: dict) -> list[dict]:
    area = GRID_SIZE_METERS * GRID_SIZE_METERS
    out = []
    for idx, ((gx, gy), seg) in enumerate(sorted(raw_segments.items()), start=1):
        center_x = (seg['min_x'] + seg['max_x']) / 2
        center_y = (seg['min_y'] + seg['max_y']) / 2
        center_z = (seg['min_z'] + seg['max_z']) / 2
        height_range = seg['max_z'] - seg['min_z']
        point_count = int(seg['point_count'])
        out.append({
            'segment_id': f'S{idx:03d}',
            'grid_x': gx,
            'grid_y': gy,
            'min_x': seg['min_x'],
            'max_x': seg['max_x'],
            'min_y': seg['min_y'],
            'max_y': seg['max_y'],
            'min_z': seg['min_z'],
            'max_z': seg['max_z'],
            'center_x': center_x,
            'center_y': center_y,
            'center_z': center_z,
            'point_count': point_count,
            'density': point_count / area,
            'height_range': height_range,
        })
    return out


def risk_level(score: int) -> str:
    if score >= 85:
        return 'critical'
    if score >= 65:
        return 'high'
    if score >= 40:
        return 'medium'
    return 'low'


def graph_exit_distances(graph: nx.Graph) -> tuple[dict[str, float], float]:
    exit_nodes = [node for node in graph.nodes if graph.degree(node) <= 1]
    if not exit_nodes:
        return {}, 0.0
    distances = nx.multi_source_dijkstra_path_length(graph, exit_nodes, weight='weight')
    max_distance = max(distances.values(), default=0.0)
    return distances, max_distance


def graph_cycle_nodes(graph: nx.Graph) -> set[str]:
    nodes = set()
    for cycle in nx.cycle_basis(graph):
        nodes.update(cycle)
    return nodes


def make_geometry_risk(segments: list[dict], graph: nx.Graph) -> list[dict]:
    density_max = max((s['density'] for s in segments), default=1.0)
    exit_distances, max_exit_distance = graph_exit_distances(graph)
    articulation_nodes = set(nx.articulation_points(graph)) if graph.number_of_nodes() else set()
    cycle_nodes = graph_cycle_nodes(graph)
    risks = []
    for s in segments:
        degree = graph.degree(s['segment_id'])
        x_span = s['max_x'] - s['min_x']
        y_span = s['max_y'] - s['min_y']
        width_m = min(x_span, y_span)
        length_m = max(x_span, y_span)
        narrow_passage_risk = 1.0 - min(1.0, width_m / NARROW_PASSAGE_WIDTH_M) if width_m > 0 else 1.0
        distance = exit_distances.get(s['segment_id'])
        exit_distance_risk = (distance / max_exit_distance) if distance is not None and max_exit_distance > 0 else 0.5
        single_connection_risk = 1.0 if degree <= 1 else 0.0
        limited_alternative_route_risk = 1.0 if s['segment_id'] not in cycle_nodes and degree <= 2 else 0.0
        critical_passage_risk = 1.0 if s['segment_id'] in articulation_nodes else 0.0
        sparse_geometry_risk = 1.0 - min(1.0, s['density'] / density_max) if density_max > 0 else 1.0
        weighted = (
            0.20 * narrow_passage_risk
            + 0.20 * exit_distance_risk
            + 0.15 * single_connection_risk
            + 0.20 * limited_alternative_route_risk
            + 0.20 * critical_passage_risk
            + 0.05 * sparse_geometry_risk
        )
        score = int(round(100 * weighted))
        reasons = []
        if narrow_passage_risk >= 0.55:
            reasons.append('Dar gecit olasiligi yuksek')
        elif narrow_passage_risk >= 0.25:
            reasons.append('Gecit genisligi sinirli')
        if exit_distance_risk >= 0.65:
            reasons.append('Cikisa uzak bolge')
        if single_connection_risk == 1.0:
            reasons.append('Tek baglantili segment')
        elif limited_alternative_route_risk == 1.0:
            reasons.append('Alternatif rotasi az olan segment')
        if critical_passage_risk == 1.0:
            reasons.append('Gocuk senaryosunda kritik gecis noktasi')
        if sparse_geometry_risk >= 0.60:
            reasons.append('Nokta yogunlugu dusuk / seyrek geometri')
        if not reasons:
            reasons.append('Geometri ve baglanti acisindan dusuk risk')
        risks.append({
            'segment_id': s['segment_id'],
            'geometry_risk': score,
            'risk_level': risk_level(score),
            'reasons': reasons,
            'components': {
                'narrow_passage_risk': round(float(narrow_passage_risk), 4),
                'exit_distance_risk': round(float(exit_distance_risk), 4),
                'single_connection_risk': round(float(single_connection_risk), 4),
                'limited_alternative_route_risk': round(float(limited_alternative_route_risk), 4),
                'critical_passage_risk': round(float(critical_passage_risk), 4),
                'sparse_geometry_risk': round(float(sparse_geometry_risk), 4),
            },
            'metrics': {
                'degree': int(degree),
                'length_m': round(float(length_m), 3),
                'width_m': round(float(width_m), 3),
                'height_m': round(float(s['height_range']), 3),
                'distance_to_nearest_exit_m': round(float(distance), 3) if distance is not None else None,
                'point_density': round(float(s['density']), 4),
            },
        })
    return risks


def build_graph_object(segments: list[dict]) -> nx.Graph:
    by_grid = {(s['grid_x'], s['grid_y']): s for s in segments}
    graph = nx.Graph()

    for s in segments:
        graph.add_node(
            s['segment_id'],
            center=[s['center_x'], s['center_y'], s['center_z']],
            grid=[s['grid_x'], s['grid_y']],
            point_count=s['point_count'],
        )

    for s in segments:
        for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            n = by_grid.get((s['grid_x'] + dx, s['grid_y'] + dy))
            if not n:
                continue
            dist = math.dist(
                [s['center_x'], s['center_y'], s['center_z']],
                [n['center_x'], n['center_y'], n['center_z']],
            )
            graph.add_edge(s['segment_id'], n['segment_id'], weight=round(dist, 3))

    return graph


def serialize_graph(graph: nx.Graph) -> dict:
    return {
        'nodes': [{'id': node, **attrs} for node, attrs in graph.nodes(data=True)],
        'edges': [{'source': u, 'target': v, **attrs} for u, v, attrs in graph.edges(data=True)],
    }


def approximate_main_path(graph: nx.Graph) -> set[str]:
    if graph.number_of_nodes() == 0:
        return set()
    largest_component = max(nx.connected_components(graph), key=len)
    subgraph = graph.subgraph(largest_component)
    candidates = [node for node in subgraph.nodes if subgraph.degree(node) <= 1]
    if not candidates:
        candidates = list(subgraph.nodes)
    start = candidates[0]
    lengths = nx.single_source_dijkstra_path_length(subgraph, start, weight='weight')
    farthest = max(lengths, key=lengths.get)
    lengths = nx.single_source_dijkstra_path_length(subgraph, farthest, weight='weight')
    other = max(lengths, key=lengths.get)
    return set(nx.shortest_path(subgraph, farthest, other, weight='weight'))


def make_semantic_segments(segments: list[dict], graph: nx.Graph, geometry_risk: list[dict]) -> list[dict]:
    risk_by_id = {item['segment_id']: item for item in geometry_risk}
    main_path = approximate_main_path(graph)
    out = []
    for s in segments:
        risk_value = risk_by_id[s['segment_id']]['geometry_risk']
        connected = sorted(graph.neighbors(s['segment_id']))
        degree = len(connected)
        is_exit_candidate = degree <= 1
        is_risky = risk_value >= 65

        if is_exit_candidate:
            segment_type = 'exit_point'
            role = 'exit_or_dead_end_candidate'
        elif s['segment_id'] in main_path:
            segment_type = 'main_tunnel'
            role = 'main_route'
        elif degree >= 3:
            segment_type = 'junction'
            role = 'gallery_connection'
        else:
            segment_type = 'side_gallery'
            role = 'secondary_route'

        length_m = max(s['max_x'] - s['min_x'], s['max_y'] - s['min_y'])
        out.append({
            'segment_id': s['segment_id'],
            'name': f"{segment_type.replace('_', ' ').title()} {s['segment_id']}",
            'type': segment_type,
            'role': role,
            'center': [round(s['center_x'], 3), round(s['center_y'], 3), round(s['center_z'], 3)],
            'length_m': round(float(length_m), 3),
            'connected_segments': connected,
            'approx_position': {'grid_x': s['grid_x'], 'grid_y': s['grid_y']},
            'bounds': {
                'min_x': round(s['min_x'], 3),
                'max_x': round(s['max_x'], 3),
                'min_y': round(s['min_y'], 3),
                'max_y': round(s['max_y'], 3),
                'min_z': round(s['min_z'], 3),
                'max_z': round(s['max_z'], 3),
            },
            'point_count': s['point_count'],
            'density': round(float(s['density']), 4),
            'height_range': round(float(s['height_range']), 4),
            'geometry_risk': risk_value,
            'is_risky': is_risky,
            'is_exit_candidate': is_exit_candidate,
            'classification_source': 'auto_xy_grid_graph',
        })
    return out


def make_segment_metadata(map_segments: list[dict]) -> list[dict]:
    metadata = []
    for segment in map_segments:
        bounds = segment['bounds']
        x_span = bounds['max_x'] - bounds['min_x']
        y_span = bounds['max_y'] - bounds['min_y']
        length_m = max(x_span, y_span)
        width_m = min(x_span, y_span)
        height_m = bounds['max_z'] - bounds['min_z']
        metadata.append({
            'segment_id': segment['segment_id'],
            'segment_name': segment['name'],
            'segment_type': segment['type'],
            'center_position': segment['center'],
            'length_m': round(float(length_m), 3),
            'width_m': round(float(width_m), 3),
            'height_m': round(float(height_m), 3),
            'connected_segments': segment['connected_segments'],
            'is_exit': bool(segment['is_exit_candidate']),
            'is_blocked': False,
            'base_geometry_risk': segment['geometry_risk'],
        })
    return metadata

## Ana Pipeline

Bu hucre LAS dosyalarini bir kez okur ve ayni geciste:

- 500k / 1M PLY dosyalarini yazar
- X-Y segment istatistiklerini toplar
- Sonradan JSON ciktilari uretir

Bu islem veri buyuklugune ve Google Drive hizina gore uzun surebilir.

In [ ]:
def run_pipeline():
    strides = {
        name: compute_stride(bounds.point_count, target)
        for name, target in QUALITY_LEVELS.items()
    }
    actual_counts = {
        name: count_sampled_points(las_paths, stride)
        for name, stride in strides.items()
    }

    ply_paths = {
        name: POINTCLOUD_DIR / PLY_OUTPUT_FILENAMES[name]
        for name in QUALITY_LEVELS
    }

    handles = {}
    try:
        for name, path in ply_paths.items():
            handle = path.open('wb')
            write_binary_ply_header(handle, actual_counts[name])
            handles[name] = handle
            print(f'{name}: target={QUALITY_LEVELS[name]:,}, stride={strides[name]}, actual={actual_counts[name]:,}, path={path}')

        raw_segments = {}
        global_index = 0

        for path in tqdm(las_paths, desc='LAS files'):
            with laspy.open(path) as reader:
                for points in reader.chunk_iterator(CHUNK_SIZE):
                    x = np.asarray(points.x, dtype=np.float64) - origin['x']
                    y = np.asarray(points.y, dtype=np.float64) - origin['y']
                    z = np.asarray(points.z, dtype=np.float64) - origin['z']
                    n = x.size

                    update_segments(raw_segments, x, y, z)

                    for name, stride in strides.items():
                        first = (-global_index) % stride
                        if first < n:
                            selected = np.column_stack([
                                x[first::stride],
                                y[first::stride],
                                z[first::stride],
                            ]).astype('<f4', copy=False)
                            handles[name].write(selected.tobytes(order='C'))

                    global_index += n

        segments = finalize_segments(raw_segments)
        graph = build_graph_object(segments)
        graph_json = serialize_graph(graph)
        geometry_risk = make_geometry_risk(segments, graph)
        map_segments = make_semantic_segments(segments, graph, geometry_risk)
        segment_metadata = make_segment_metadata(map_segments)

        save_json(SEGMENT_DIR / 'map_segments.json', map_segments)
        save_json(SEGMENT_DIR / 'segment_metadata.json', segment_metadata)
        save_json(RISK_DIR / 'geometry_risk.json', geometry_risk)
        save_json(GRAPH_DIR / 'mine_graph.json', graph_json)

        summary = {
            'input_files': [str(p) for p in las_paths],
            'total_points_processed': bounds.point_count,
            'bounds': asdict(bounds),
            'origin_global_shift': origin,
            'grid_size_meters': GRID_SIZE_METERS,
            'segment_count': len(segments),
            'graph_node_count': len(graph_json['nodes']),
            'graph_edge_count': len(graph_json['edges']),
            'quality_levels': {
                name: {
                    'target_points': QUALITY_LEVELS[name],
                    'stride': strides[name],
                    'actual_points': actual_counts[name],
                    'output_path': str(ply_paths[name]),
                }
                for name in QUALITY_LEVELS
            },
            'outputs': {
                'global_shift': str(POINTCLOUD_DIR / 'global_shift.json'),
                'map_segments': str(SEGMENT_DIR / 'map_segments.json'),
                'segment_metadata': str(SEGMENT_DIR / 'segment_metadata.json'),
                'geometry_risk': str(RISK_DIR / 'geometry_risk.json'),
                'mine_graph': str(GRAPH_DIR / 'mine_graph.json'),
            },
        }
        save_json(SUMMARY_DIR / 'digital_twin_lidar_pipeline_summary.json', summary)
        return summary
    finally:
        for handle in handles.values():
            handle.close()


summary = run_pipeline()
print(json.dumps(summary, indent=2)[:5000])

## Ciktilari Kontrol Et

In [ ]:
for path in sorted(POINTCLOUD_DIR.glob('*.ply')):
    print(path.name, round(path.stat().st_size / 1_000_000, 2), 'MB')

for path in sorted(SEGMENT_DIR.glob('*.json')):
    print(path.name, round(path.stat().st_size / 1_000_000, 3), 'MB')

for path in sorted(GRAPH_DIR.glob('*.json')):
    print(path.name, round(path.stat().st_size / 1_000_000, 3), 'MB')

for path in sorted(RISK_DIR.glob('*.json')):
    print(path.name, round(path.stat().st_size / 1_000_000, 3), 'MB')

print('Summary:', SUMMARY_DIR / 'digital_twin_lidar_pipeline_summary.json')

## Frontend ve CloudCompare Icin Oneri

Frontend'de once hizli modeli kullan:

```text
backend/data_processed/pointcloud/tunnel_preview_500k.ply
```

Daha net frontend demosu icin bunu kullan:

```text
backend/data_processed/pointcloud/tunnel_downsampled.ply
```

CloudCompare icin daha detayli gorsel gerekirse `QUALITY_LEVELS` ve `PLY_OUTPUT_FILENAMES` sozluklerine 5M veya 10M seviyesinde yeni bir cikti eklenebilir.